# ArtiBench tutorial

What a suite hands your method, and what your method hands back.

The metrics contain no ICP, no Procrustes fit and no rotation search, so the two frames have to
match before scoring.


## Notes

- Run from `ArtiBench/`. Everything here works on the two objects in `examples/`, so no
  suite download is needed; point `SUITE` at a real suite and the same code runs unchanged.
- Needs the base install plus `pytorch3d`.
- `examples/mini_suite/` is a two-object miniature of a released suite: same directory layout, same
  manifest schema. `examples/predictions/` is a real Artisan run on those two objects.
- The two objects are picked so you can read their orientation off the numbers: a floor cabinet
  whose doors sit on one face, and a wheeled bin with a lid on top.


## 0. Check the install

Scores the two objects in `examples/predictions/` against ground truth and compares the result
to a stored fixture. A mismatch means the install is wrong.


In [1]:
!bash scripts/smoke_test.sh


==> evaluating examples/predictions against examples/mini_suite/gt
[Paired Eval] Found 2 pair(s)  (recon_norm=mesh)   gt_total=2  missing=0
[timing:00188] parse=0.00s C1=0.05s C2=6.45s | total=8.02s
[timing:00620] parse=0.00s C1=0.10s C2=0.58s | total=0.86s
Eval pairs: 100%|##########| 2/2 [00:08<00:00,  4.44s/it]
  [whole_mesh] rest_overall_cd=0.0002
  [part_meshes] count_accuracy=1.0000, rest_per_part_cd=0.0005, rest_per_part_miou=0.9473, rest_per_part_giou=0.9472
  [articulation] type_accuracy=1.0000, axis_error_rad_mean=0.0004, origin_error_mean=0.0123, limit_error_mean=0.4317, success_rate=0.8333, articulated_per_part_cd=0.0608, articulated_per_part_miou=0.8375, articulated_per_part_giou=0.8136

[Done] 2 / 2 pairs evaluated  (coverage=100.0%, pred_missing=0)

==> comparing against examples/expected/results.json  (atol 1e-3, rtol 0.05)
OK — 13 metrics match.

Install verified. Next: download a suite and evaluate your own predictions.


In [2]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import trimesh

from artibench.core.urdf_parser import parse_urdf
from artibench.core.forward_kinematics import compute_fk

REPO = Path.cwd()
SUITE = REPO / 'examples' / 'mini_suite'      # or $ARTIBENCH_DATA_ROOT/<suite>
PRED = REPO / 'examples' / 'predictions'
assert (SUITE / 'manifest.json').is_file(), f'run from the repo root; no suite at {SUITE}'

MANIFEST = json.loads((SUITE / 'manifest.json').read_text())
ENTRY = {m['model_name']: m for m in MANIFEST['models']}
print(f"{MANIFEST['num_models']} objects, {MANIFEST['num_views_per_model']} views each, "
      f"prompt_seed={MANIFEST['prompt_seed']}")
for name, e in ENTRY.items():
    print(f'  {name}  n_parts={e["n_parts"]}  prompt view {e["prompt_view_index"]}')


2 objects, 4 views each, prompt_seed=0
  00188  n_parts=4  prompt view 3
  00620  n_parts=3  prompt view 1


## 1. What the suite hands you

`manifest.json` is the entry point; every path inside it is suite-relative. Per object:

| | |
|---|---|
| `whole_mesh/<oid>/mesh.obj` | the assembled object, for whole-mesh methods |
| `part_meshes/<oid>/` | the same object already split, for part-mesh methods |
| `images/<oid>/` | four rendered views plus per-view part-label EXRs; `<oid>_prompt.png` is the one view every image metric scores against |
| `cached_link_point_clouds/<oid>/` | pre-sampled per-link surface points, so ground truth does not get re-sampled at eval time |
| `gt/<oid>/` | `mobility.urdf` and its meshes — what you are scored against |

`in_the_wild` has no `gt/` and no `cached_link_point_clouds/`: it has no 3D ground truth by design,
so the paired C1/C2 metrics are skipped for it.


In [3]:
OID = '00620'   # the cabinet
e = ENTRY[OID]
for key in ('whole_mesh_path', 'part_mesh_dir', 'image_dir', 'gt_pcl_dir', 'model_dir'):
    p = SUITE / e[key]
    n = len(list(p.rglob('*'))) if p.is_dir() else 1
    print(f'  {key:<16} {e[key]:<28} {"exists" if p.exists() else "MISSING":<8} {n} file(s)')


  whole_mesh_path  whole_mesh/00620/mesh.obj    exists   1 file(s)
  part_mesh_dir    part_meshes/00620            exists   7 file(s)
  image_dir        images/00620                 exists   11 file(s)
  gt_pcl_dir       cached_link_point_clouds/00620 exists   2 file(s)
  model_dir        gt/00620                     exists   77 file(s)


## 2. The input mesh is Y-up


In [4]:
mesh = trimesh.load(SUITE / e['whole_mesh_path'], force='mesh', process=False)
V = np.asarray(mesh.vertices)
extent = V.max(0) - V.min(0)
print(f'whole_mesh   {len(V)} verts, {len(mesh.faces)} faces')
print(f'             extent X={extent[0]:.2f}  Y={extent[1]:.2f}  Z={extent[2]:.2f}'
      f"   -> tallest axis {'XYZ'[int(np.argmax(extent))]}")
print(f'             visual: {type(mesh.visual).__name__}, uv={getattr(mesh.visual, "uv", None) is not None}')


whole_mesh   2639 verts, 3865 faces
             extent X=1.47  Y=2.00  Z=0.96   -> tallest axis Y
             visual: TextureVisuals, uv=True


The suites use PartNet-Mobility's convention: **Y-up, +Z front**. Submissions are judged in the
world frame after forward kinematics — section 4.


## 3. Sampling: points, colours, part ids

The geometry metrics work on sampled surface points, not vertices. Sampling the whole mesh and the
part meshes the same way is what makes them comparable.

Colour comes from the mesh's own texture or material. Points are not normalised here;
`--input-modality` decides that at evaluation time.


In [5]:
def sample_xyzrgb(mesh, n_points, seed=0):
    """Surface points with per-point colour, in the mesh's own frame and scale."""
    rng = np.random.default_rng(seed)
    pts, face_idx = trimesh.sample.sample_surface(mesh, n_points, seed=int(rng.integers(1 << 31)))
    try:
        rgb = np.asarray(mesh.visual.to_color().face_colors)[face_idx][:, :3]
    except Exception:
        rgb = np.full((len(pts), 3), 200, np.uint8)
    return np.asarray(pts), rgb


xyz, rgb = sample_xyzrgb(mesh, 8192)
print(f'whole mesh   xyz {xyz.shape} {xyz.dtype}   rgb {rgb.shape} {rgb.dtype}')
print(f'             centroid {np.round(xyz.mean(0), 3)}   radius {np.linalg.norm(xyz - xyz.mean(0), axis=1).max():.3f}')

part_dir = SUITE / e['part_mesh_dir']
part_files = sorted(p for p in part_dir.glob('part_*.obj'))
print(f'\npart_meshes  {len(part_files)} parts, and n_parts in the manifest says {e["n_parts"]}')
for f in part_files:
    pm = trimesh.load(f, force='mesh', process=False)
    pv = np.asarray(pm.vertices)
    print(f'  {f.name:<16} {len(pv):>6} verts   centroid {np.round(pv.mean(0), 3)}')


whole mesh   xyz (8192, 3) float64   rgb (8192, 3) uint8
             centroid [-0.001  0.01  -0.026]   radius 1.304

part_meshes  3 parts, and n_parts in the manifest says 3
  part_000.obj       1476 verts   centroid [ 0.012 -0.914 -0.023]
  part_001.obj        641 verts   centroid [0.078 0.679 0.442]
  part_002.obj        522 verts   centroid [-0.079  0.64   0.445]


## 4. What you hand back: Z-up, facing −X

**Forward-kinematics your URDF at zero joint state. The object must come out Z-up in world space,
facing −X, at the scale of your input mesh.**

You do not have to re-author your geometry to get there. Do what the ground truth does: add an empty
`base` link and connect it to your model's root link with one `fixed` joint carrying the rotation in
its `rpy`. Your meshes and every articulated joint stay exactly as your model predicted them — a
door's hinge stays in the door gap — because the whole chain hangs below that one joint and rotates
with it.

`describe()` below is the whole check.


In [6]:
def posed_geometry(urdf_path):
    """FK at zero joint state; return the object, its poses, and world-space vertices per link."""
    obj = parse_urdf(str(urdf_path), prefer_collision=False)
    poses = compute_fk(obj, {})
    out = {}
    for link in obj.links:
        T = poses.get(link.name)
        if T is None:
            continue
        chunks = []
        origins = link.origins or [np.eye(4)] * len(link.mesh_paths)
        for rel, org in zip(link.mesh_paths, origins):
            f = Path(obj.urdf_dir) / rel
            if not f.is_file():
                continue
            v = np.asarray(trimesh.load(f, force='mesh', process=False).vertices)
            O = np.asarray(org) if org is not None else np.eye(4)
            v = (O[:3, :3] @ v.T).T + O[:3, 3]
            chunks.append((T[:3, :3] @ v.T).T + T[:3, 3])
        if chunks:
            out[link.name] = np.vstack(chunks)
    return obj, poses, out


def describe(urdf_path):
    obj, poses, per_link = posed_geometry(urdf_path)
    V = np.vstack(list(per_link.values()))
    lo, hi = V.min(0), V.max(0)
    ext, ctr = hi - lo, (lo + hi) / 2
    print(f"world AABB   X={ext[0]:.2f}  Y={ext[1]:.2f}  Z={ext[2]:.2f}"
          f"   tallest axis: {'XYZ'[int(np.argmax(ext))]}")
    for name, v in sorted(per_link.items()):
        c = v.mean(0) - ctr
        print(f'  {name:<10} centroid  X={c[0]:+.3f}  Y={c[1]:+.3f}  Z={c[2]:+.3f}')
    for j in obj.joints:
        if j.type == 'fixed':
            continue
        a = np.asarray(j.axis, float)
        a = a / np.linalg.norm(a)
        aw = poses[j.child_link][:3, :3] @ a
        d = int(np.argmax(np.abs(aw)))
        sign = '+' if aw[d] > 0 else '-'
        print(f'  {j.name:<10} {j.type:<10} axis in file {np.round(a, 2)}'
              f'  ->  world {np.round(aw, 2)}  ({sign}{"XYZ"[d]})')
    return ext, ctr


describe(SUITE / 'gt' / OID / 'mobility.urdf');


world AABB   X=0.96  Y=1.47  Z=2.00   tallest axis: Z
  link_0     centroid  X=+0.023  Y=-0.012  Z=-0.914
  link_1     centroid  X=-0.442  Y=-0.078  Z=+0.679
  link_2     centroid  X=-0.445  Y=+0.079  Z=+0.640
  joint_1    revolute   axis in file [0. 1. 0.]  ->  world [-0.  0.  1.]  (+Z)
  joint_2    revolute   axis in file [ 0. -1.  0.]  ->  world [ 0. -0. -1.]  (-Z)


Against the Y-up extents in section 2: Z is now the longest axis, both doors sit at X ≈ −0.44
separated in Y, and both hinge axes are ±Z — vertical hinges on the −X face.

The axes are written `[0, ±1, 0]` in the file and become vertical only once the root joint is
applied. **A joint axis in a URDF is expressed in its own frame, not in world space.**


In [7]:
describe(SUITE / 'gt' / '00188' / 'mobility.urdf');


world AABB   X=1.43  Y=1.29  Z=2.00   tallest axis: Z
  link_0     centroid  X=+0.444  Y=-0.022  Z=+0.464
  link_1     centroid  X=+0.416  Y=-0.004  Z=+0.796
  link_2     centroid  X=+0.316  Y=+0.616  Z=-0.796
  link_3     centroid  X=+0.316  Y=-0.616  Z=-0.796
  joint_1    revolute   axis in file [-1.  0.  0.]  ->  world [0. 1. 0.]  (+Y)
  joint_2    continuous axis in file [1. 0. 0.]  ->  world [-0. -1.  0.]  (-Y)
  joint_3    continuous axis in file [1. 0. 0.]  ->  world [-0. -1.  0.]  (-Y)


The lid is near the top (Z ≈ +0.80) and toward the back (X ≈ +0.42), hinged about world Y so it
opens toward −X. The wheels sit on the ground, separated along Y, sharing a world-Y axle. All three
are `[±1, 0, 0]` in the file.


## 5. Cached ground-truth point clouds

`cached_link_point_clouds/<oid>/link_pcls.npz` holds pre-sampled per-link surface points. They are
the authoritative ground-truth samples the geometry metrics consume — caching them removes the
run-to-run variation of surface sampling, so two evaluations of the same prediction agree.

They are stored in the **link-local** frame: each primitive's URDF `<origin>` is applied, but FK is
not. Evaluation articulates them with `compute_fk()` to whatever joint state a metric needs.


In [8]:
pcl = np.load(SUITE / e['gt_pcl_dir'] / 'link_pcls.npz')
for link_name in pcl.files:
    pts = pcl[link_name]
    print(f'  {link_name:<10} {pts.shape}  centroid {np.round(pts.mean(0), 3)}   (link-local)')


  link_0     (8192, 3)  centroid [-0.004  0.016 -0.131]   (link-local)
  link_1     (8192, 3)  centroid [-0.357  0.024  0.017]   (link-local)
  link_2     (8192, 3)  centroid [0.359 0.025 0.017]   (link-local)


## 6. A mis-oriented submission

`scripts/check_urdf_conventions.py` audits structure; `eval_self` never looks at ground
truth. Neither checks the frame. Turn the example prediction and read the same numbers.


In [9]:
import shutil
import xml.etree.ElementTree as ET


def rotate_submission(urdf_path, dst_dir, degrees, about='z'):
    """Copy a submission and turn its root fixed joint — i.e. break the frame rule on purpose."""
    urdf_path, dst_dir = Path(urdf_path), Path(dst_dir)
    shutil.rmtree(dst_dir, ignore_errors=True)
    shutil.copytree(urdf_path.parent, dst_dir)
    tree = ET.parse(dst_dir / urdf_path.name)
    for j in tree.getroot().findall('joint'):
        if j.get('type') != 'fixed':
            continue
        o = j.find('origin')
        rpy = np.array([float(v) for v in o.get('rpy', '0 0 0').split()])
        rpy['xyz'.index(about)] += np.radians(degrees)
        o.set('rpy', ' '.join(f'{v:.15f}' for v in rpy))
        break
    tree.write(dst_dir / urdf_path.name, encoding='utf-8', xml_declaration=True)
    return dst_dir / urdf_path.name


print('as submitted')
describe(PRED / OID / 'mobility.urdf')

broken = rotate_submission(PRED / OID / 'mobility.urdf', '/tmp/rotated/' + OID, 90, 'z')
print('\nafter a 90 degree error about Z')
describe(broken);


as submitted
world AABB   X=0.96  Y=1.47  Z=2.00   tallest axis: Z
  link_0     centroid  X=-0.445  Y=+0.079  Z=+0.640
  link_1     centroid  X=-0.443  Y=-0.078  Z=+0.668
  link_2     centroid  X=+0.045  Y=+0.000  Z=-0.922
  joint_2_0  revolute   axis in file [-0.  1. -0.]  ->  world [0. 0. 1.]  (+Z)
  joint_2_1  revolute   axis in file [ 0.  1. -0.]  ->  world [ 0. -0.  1.]  (+Z)

after a 90 degree error about Z
world AABB   X=1.47  Y=0.96  Z=2.00   tallest axis: Z
  link_0     centroid  X=-0.079  Y=-0.445  Z=+0.640
  link_1     centroid  X=+0.078  Y=-0.443  Z=+0.668
  link_2     centroid  X=-0.000  Y=+0.045  Z=-0.922
  joint_2_0  revolute   axis in file [-0.  1. -0.]  ->  world [-0.  0.  1.]  (+Z)
  joint_2_1  revolute   axis in file [ 0.  1. -0.]  ->  world [0. 0. 1.]  (+Z)


The doors move from X ≈ −0.44 to Y ≈ −0.44 and the AABB's X and Y swap. Z is still the tallest
axis, so an upright check passes.

Scored, `axis_error_rad_mean` goes to ≈ 1.5708 (π/2, the largest the metric reports) and
`rest_overall_cd` jumps by orders of magnitude, while `type_accuracy`, `count_accuracy` and CLIP
barely move. Compare your world-space AABB against `gt/<oid>/mobility.urdf` before submitting;
the cell above is that check.


## 7. The rest of the submission format

The README is the reference for the format. Here is what it looks like in a file, and how to
check your own against it:


In [10]:
pred_urdf = PRED / OID / 'mobility.urdf'
root = ET.parse(pred_urdf).getroot()
links = [l.get('name') for l in root.findall('link')]
joints = [(j.get('name'), j.get('type'), j.find('parent').get('link'), j.find('child').get('link'))
          for j in root.findall('joint')]
print(f'{pred_urdf.relative_to(REPO)}')
print(f'  links  {links}')
for name, jtype, parent, child in joints:
    limit = ET.parse(pred_urdf).getroot().find(f"joint[@name='{name}']/limit")
    has = 'limit' if limit is not None else 'no limit'
    print(f'  {name:<18} {jtype:<11} {parent:<10} -> {child:<10} {has}')


examples/predictions/00620/mobility.urdf
  links  ['base', 'link_0', 'link_1', 'link_2']
  joint_2_0          revolute    link_2     -> link_0     limit
  joint_2_1          revolute    link_2     -> link_1     limit
  joint_base_link_2  fixed       base       -> link_2     no limit
